# TRELLIS.2 + FLUX.2 notebook quick start

1. Select the Jupyter kernel for the `phase2_3d` conda environment and confirm the project root is `~/OEC`.
2. Restart the kernel if a setup cell says an already-imported package was upgraded. Then run the notebook from the first cell in order; do not jump directly into a model studio in a fresh kernel.
3. Setup cells use the active Python interpreter (`sys.executable`) and must pass OEC StorageManager admission before installing dependencies, compiling extensions, or downloading model weights. StorageManager checks the configured allocation, intact reserve and actual filesystem headroom; do not bypass a denied preflight.
4. TRELLIS launches on port 7862. FLUX launches on port 7863. The Gradio share link is disabled by default.

### SSH tunnels

Print the compute node name from the notebook with `!hostname`. Run the appropriate command on your local computer and replace `<COMPUTE_NODE>` with that exact host name:

```bash
ssh -N -L 5402:<COMPUTE_NODE>:8888 rahkus@login.npsf.cdac.in
ssh -N -L 7862:<COMPUTE_NODE>:7862 rahkus@login.npsf.cdac.in
ssh -N -L 7863:<COMPUTE_NODE>:7863 rahkus@login.npsf.cdac.in
```

Keep these tunnels in separate terminals while using Jupyter and the Gradio interfaces.


In [1]:
# 1. THE FROZEN RULE: Core Config First
# Keep shared-cache purging disabled unless an operator has verified no concurrent loaders.
ENABLE_DISK_PURGE = False
import core_config
print("[SYSTEM] Core proxies, storage mapping, and HF Auth executed automatically.")
from core_config import SYSTEM_DIRS

# 2. ISOLATE GPU: Lock this entire notebook to a specific GPU to prevent VRAM collisions
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # Change to 0, 1, 2, or 3
print(f"[SYSTEM] Notebook strictly locked to GPU: {os.environ['CUDA_VISIBLE_DEVICES']}")


[CORE] Secrets & Config loaded successfully from /nlsasfs/home/lmedge/rahkus/.env
[CORE] Cleaned trash at /nlsasfs/home/lmedge/rahkus/.local/share/Trash/files
[CORE] Cleaned trash at /nlsasfs/home/lmedge/rahkus/.local/share/Trash/info
[CORE] Cleaned trash at /nlsasfs/home/lmedge/rahkus/.jupyter/lab/workspaces
[SYSTEM] Core proxies, storage mapping, and HF Auth executed automatically.
[SYSTEM] Notebook strictly locked to GPU: 0


In [2]:
# Replaced the old hard-coded "kill 9 <PID>" command with a read-only process listing.
# Inspect the owner/command and stop only a verified stale process manually.
!ps -u "$USER" -o pid,etime,cmd | grep -E '[g]radio|[p]ipeline_trellis' || true

/bin/bash: line 1: kill: (9) - No such process
/bin/bash: line 1: kill: (2386856) - No such process


In [3]:
!nvidia-smi

Sat Aug 22 23:21:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:07:00.0 Off |                    0 |
| N/A   29C    P0             55W /  400W |       4MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
# Verify notebook UI dependencies without upgrading packages during model setup.
import importlib.metadata

for package in ("jupyter", "ipywidgets", "ipykernel"):
    try:
        print(f"{package}=={importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package}=MISSING (install only in the intended phase2_3d environment after storage preflight)")


# Step 1: TRELLIS.2 (Setup)
Prepare TRELLIS.2 Python and native dependencies. The setup first checks OEC storage admission and existing extensions; it will not delete or replace an existing source checkout automatically.

In [ ]:
import os
import sys
import subprocess
import shutil
import importlib.util
from core_config import SYSTEM_DIRS
from storage_manager import managed_storage_operation, storage_preflight_path

FORCE_RECOMPILE = False
if FORCE_RECOMPILE:
    raise RuntimeError('Force recompile is disabled in the shared environment; preserve existing extension packages and source checkouts.')

# All dependency installs/builds are performed on the compute node, but they
# still consume the shared user quota. Admit the setup phase before pip/conda.
setup_budget_gb = max(
    1.0,
    float(os.environ.get("AIEROS_TRELLIS_SETUP_BUDGET_GB", "8.0")),
)
setup_probe_path = SYSTEM_DIRS["cache"]
while not os.path.isdir(setup_probe_path) and os.path.dirname(setup_probe_path) != setup_probe_path:
    setup_probe_path = os.path.dirname(setup_probe_path)
ok, reason = storage_preflight_path(
    setup_probe_path,
    required_gb=setup_budget_gb,
    require_fresh_quota=True,
    minimum_free_gb=0.5,
)
if not ok:
    raise RuntimeError(f"TRELLIS setup denied by storage policy: {reason}")

def _trellis_setup_storage_admission(required_gb=None):
    required_gb = setup_budget_gb if required_gb is None else required_gb
    setup_probe_path = SYSTEM_DIRS["cache"]
    while not os.path.isdir(setup_probe_path) and os.path.dirname(setup_probe_path) != setup_probe_path:
        setup_probe_path = os.path.dirname(setup_probe_path)
    ok, reason = storage_preflight_path(
        setup_probe_path,
        required_gb=required_gb,
        require_fresh_quota=True,
        minimum_free_gb=0.5,
    )
    if not ok:
        raise RuntimeError(f"TRELLIS setup write denied: {reason}")

def run_cmd(cmd, cwd=None):
    print(f"Running: {' '.join(cmd)}")
    tokens = [str(part) for part in cmd]
    is_pip_install = len(tokens) >= 4 and tokens[1:4] == ["-m", "pip", "install"]
    is_git_clone = len(tokens) >= 2 and tokens[0] == "git" and tokens[1] == "clone"
    if not (is_pip_install or is_git_clone):
        subprocess.run(cmd, cwd=cwd, check=True)
        return
    _trellis_setup_storage_admission()
    with managed_storage_operation(
        "models", required_gb=setup_budget_gb,
        require_fresh_quota=True, minimum_free_gb=0.5,
    ):
        _trellis_setup_storage_admission()
        subprocess.run(cmd, cwd=cwd, check=True)

def is_installed(pkg_name):
    return importlib.util.find_spec(pkg_name) is not None

os.environ.setdefault("MAX_JOBS", "1")  # Bound native build parallelism on shared compute nodes.\n\nprint("1. Verifying PyTorch 2.6.0 (CUDA 12.4)...")
_torch_probe = subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.__version__); print(torch.version.cuda)"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False,
)
_torch_ok = _torch_probe.returncode == 0 and "2.6.0" in _torch_probe.stdout and "12.4" in _torch_probe.stdout
if not _torch_ok:
    run_cmd([sys.executable, "-m", "pip", "install", "torch==2.6.0", "torchvision==0.21.0", "--index-url", "https://download.pytorch.org/whl/cu124"])
else:
    print("Existing PyTorch CUDA 12.4 runtime matches the required pin; skipping reinstall.")

_xformers_version = None
try:
    _xformers_version = importlib.metadata.version("xformers")
except importlib.metadata.PackageNotFoundError:
    pass
if _xformers_version != "0.0.29.post2":
    print("1b. Installing CUDA-enabled xFormers matched to PyTorch 2.6.0+cu124...")
    run_cmd([
        sys.executable, "-m", "pip", "install", "--force-reinstall",
        "--no-deps", "--no-cache-dir", "xformers==0.0.29.post2",
        "--index-url", "https://download.pytorch.org/whl/cu124",
    ])
else:
    print("Existing xFormers pin is correct; validating its CUDA kernels without reinstalling.")

print("Verifying installed xFormers CUDA kernels...")
_xf = subprocess.run(
    [sys.executable, "-m", "xformers.info"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False,
)
print(_xf.stdout)
if _xf.returncode != 0:
    raise RuntimeError("xformers.info failed after the pinned wheel installation.")
_xf_fields = {}
for _line in _xf.stdout.splitlines():
    if ":" in _line:
        _key, _value = _line.split(":", 1)
        _xf_fields[_key.strip()] = _value.strip()
if (
    _xf_fields.get("build.cuda_version") != "1204"
    or _xf_fields.get("pytorch.cuda") != "available"
    or _xf_fields.get("memory_efficient_attention.cutlassF-pt") != "available"
):
    raise RuntimeError(
        "xFormers CUDA validation failed. Expected CUDA 12.4 attention kernels "
        "for PyTorch 2.6.0; do not continue with TRELLIS inference."
    )
print("XFORMERS_CUDA_VALIDATION=PASS")

print("2. Verifying ML and 3D dependencies...")
run_cmd([
    sys.executable, "-m", "pip", "install", 
    "imageio", "imageio-ffmpeg", "tqdm", "easydict", "opencv-python-headless", 
    "ninja", "trimesh", "transformers==4.56.1", "gradio", "tensorboard", "pandas", 
    "lpips", "zstandard", "kornia", "timm", "einops", "safetensors", "fire", 
    "accelerate", "ipywidgets"
])
run_cmd([sys.executable, "-m", "pip", "install", "git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8"])

extensions_exist = is_installed("nvdiffrast") and is_installed("nvdiffrec_render") and is_installed("cumesh") and is_installed("flex_gemm") and is_installed("o_voxel")

if not FORCE_RECOMPILE and extensions_exist:
    print("3. ✅ 3D C++ Extensions already detected! Skipping heavy compilation.")
else:
    print("3. Compiling C++ Extensions (nvdiffrast, nvdiffrec, CuMesh, FlexGEMM, o-voxel)...")
    print("   (This will take 5-15 minutes. Please do not interrupt.)")
    extension_budget_gb = max(1.0, float(os.environ.get("AIEROS_TRELLIS_EXTENSION_BUDGET_GB", "8.0")))
    ok, reason = storage_preflight_path(
        SYSTEM_DIRS["cache"],
        required_gb=extension_budget_gb,
        require_fresh_quota=True,
        minimum_free_gb=0.5,
    )
    if not ok:
        raise RuntimeError(f"TRELLIS extension build denied by storage policy: {reason}")

    EXT_DIR = os.path.join(SYSTEM_DIRS['cache'], "trellis2_extensions")
    _trellis_setup_storage_admission(extension_budget_gb)
    with managed_storage_operation(
        "models", required_gb=extension_budget_gb,
        require_fresh_quota=True, minimum_free_gb=0.5,
    ):
        _trellis_setup_storage_admission(extension_budget_gb)
        os.makedirs(EXT_DIR, exist_ok=True)

    # Nvdiffrast
    nd_dir = os.path.join(EXT_DIR, "nvdiffrast")
    if not os.path.isdir(nd_dir):
        run_cmd(["git", "clone", "-b", "v0.4.0", "https://github.com/NVlabs/nvdiffrast.git", nd_dir])
    else:
        print(f"Reusing existing extension source checkout: {nd_dir}")
    run_cmd([sys.executable, "-m", "pip", "install", nd_dir, "--no-build-isolation"])

    # Nvdiffrec
    nr_dir = os.path.join(EXT_DIR, "nvdiffrec")
    if not os.path.isdir(nr_dir):
        run_cmd(["git", "clone", "-b", "renderutils", "https://github.com/JeffreyXiang/nvdiffrec.git", nr_dir])
    else:
        print(f"Reusing existing extension source checkout: {nr_dir}")
    run_cmd([sys.executable, "-m", "pip", "install", nr_dir, "--no-build-isolation"])

    # CuMesh (Recursive)
    cm_dir = os.path.join(EXT_DIR, "CuMesh")
    if not os.path.isdir(cm_dir):
        run_cmd(["git", "clone", "https://github.com/JeffreyXiang/CuMesh.git", cm_dir, "--recursive"])
    else:
        print(f"Reusing existing extension source checkout: {cm_dir}")
    run_cmd([sys.executable, "-m", "pip", "install", cm_dir, "--no-build-isolation"])

    # FlexGEMM (Recursive)
    fg_dir = os.path.join(EXT_DIR, "FlexGEMM")
    if not os.path.isdir(fg_dir):
        run_cmd(["git", "clone", "https://github.com/JeffreyXiang/FlexGEMM.git", fg_dir, "--recursive"])
    else:
        print(f"Reusing existing extension source checkout: {fg_dir}")
    run_cmd([sys.executable, "-m", "pip", "install", fg_dir, "--no-build-isolation"])

t2_dir = os.path.join(SYSTEM_DIRS['models'], "TRELLIS.2")
if FORCE_RECOMPILE or not os.path.exists(os.path.join(t2_dir, "setup.sh")):
    if os.path.exists(t2_dir):
        raise RuntimeError("Existing TRELLIS.2 source checkout is incomplete; refusing to delete or replace it automatically.")
    print("4. Cloning Official TRELLIS.2 Repository...")
    run_cmd(["git", "clone", "-b", "main", "https://github.com/microsoft/TRELLIS.2.git", t2_dir, "--recursive"])

if FORCE_RECOMPILE or not is_installed("o_voxel"):
    print("   Installing o-voxel...")
    run_cmd([sys.executable, "-m", "pip", "install", os.path.join(t2_dir, "o-voxel"), "--no-build-isolation"])

sys.path.insert(0, t2_dir)
print("Environment Ready! TRELLIS.2 mapped and ready.")

# Step 2: FLUX.2 (Setup)
This installs the specific diffusers patch required for FLUX.2 while protecting your PyTorch 2.6.0 environment.If the setup cell updates packages on disk, restart the kernel before continuing to either model studio.

In [ ]:
import importlib.metadata as metadata
import os
import subprocess
import sys
from packaging.version import Version

# Reuse exact, tested releases before either Gradio studio imports Diffusers.
REQUIRED_FLUX_PACKAGES = {
    "diffusers": "0.37.0",
    "huggingface-hub": "0.36.0",
    "transformers": "4.56.1",
}
missing_or_mismatched = {}
for package, required_version in REQUIRED_FLUX_PACKAGES.items():
    try:
        installed_version = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed_version = None
    if installed_version != required_version:
        missing_or_mismatched[package] = required_version
    print(f"FLUX_PACKAGE_{package}={installed_version or 'MISSING'} required={required_version}")

if missing_or_mismatched:
    # Package installation writes into this shared conda environment. Require a
    # StorageManager capacity admission and serialize with other OEC managed storage jobs.
    from storage_manager import managed_storage_operation, storage_preflight

    setup_budget_gb = max(
        1.0, float(os.environ.get("AIEROS_FLUX_SETUP_BUDGET_GB", "2.0"))
    )
    ok, reason = storage_preflight(
        "models",
        required_gb=setup_budget_gb,
        require_fresh_quota=True,
        minimum_free_gb=2.0,
    )
    print(f"FLUX_SETUP_STORAGE_ADMISSION={ok} reason={reason}")
    if not ok:
        raise RuntimeError(
            f"FLUX dependency install denied by OEC storage policy: {reason}. "
            "Free enough filesystem space for StorageManager admission, then rerun this cell."
        )

    ok, reason = storage_preflight(
        "models",
        required_gb=setup_budget_gb,
        require_fresh_quota=True,
        minimum_free_gb=2.0,
    )
    if not ok:
        raise RuntimeError(f"Storage admission denied before managed write: {reason}")
    with managed_storage_operation(
        "models",
        required_gb=setup_budget_gb,
        require_fresh_quota=True,
        minimum_free_gb=2.0,
    ):
        ok, reason = storage_preflight(
            "models",
            required_gb=setup_budget_gb,
            require_fresh_quota=True,
            minimum_free_gb=2.0,
        )
        if not ok:
            raise RuntimeError(f"Storage admission denied inside managed write: {reason}")
        subprocess.run(
            [
                sys.executable, "-m", "pip", "install",
                "diffusers==0.37.0",
                "huggingface-hub==0.36.0",
                "transformers==4.56.1",
            ],
            check=True,
        )
else:
    print("FLUX_DEPENDENCY_INSTALL=SKIPPED versions_already_match=True")

# Verify exact versions and eager imports, rather than leaving a lazy failure
# until the user clicks Generate in the studio.
subprocess.run(
    [
        sys.executable, "-c",
        (
            "import importlib.metadata as m; "
            "assert m.version('diffusers') == '0.37.0', m.version('diffusers'); "
            "assert m.version('huggingface-hub') == '0.36.0', m.version('huggingface-hub'); "
            "assert m.version('transformers') == '4.56.1', m.version('transformers'); "
            "from diffusers import DiffusionPipeline, Flux2KleinPipeline; "
            "print('FLUX_IMPORT_CHECK=PASS', "
            "'huggingface_hub='+m.version('huggingface-hub'), "
            "'diffusers='+m.version('diffusers'), "
            "'transformers='+m.version('transformers'))"
        ),
    ],
    check=True,
)
print("FLUX.2 dependencies installed or already matched and import-checked.")

_loaded_hub = sys.modules.get("huggingface_hub")
if _loaded_hub is not None:
    _loaded_hub_version = Version(getattr(_loaded_hub, "__version__", "0"))
    if _loaded_hub_version != Version("0.36.0"):
        raise RuntimeError(
            f"FLUX dependencies are now correct on disk, but this Jupyter kernel still "
            f"has huggingface-hub {_loaded_hub_version} loaded in memory. "
            "Restart the kernel, then execute the notebook from the top. "
            "Do not continue to TRELLIS/FLUX inference in this stale kernel."
        )


# Step 3: TRELLIS.2 (Interactive Studio - Image Upload)

In [ ]:
import core_config
from core_config import SYSTEM_DIRS
import os

TRELLIS_REPO_ID = "microsoft/TRELLIS.2-4B"
TRELLIS_HF_CACHE_ROOT = os.path.join(
    SYSTEM_DIRS["models"], "hub", "models--microsoft--TRELLIS.2-4B"
)
TRELLIS_SNAPSHOTS_DIR = os.path.join(TRELLIS_HF_CACHE_ROOT, "snapshots")
TRELLIS_LOCAL_CACHE_AVAILABLE = (
    os.path.isdir(TRELLIS_SNAPSHOTS_DIR) and bool(os.listdir(TRELLIS_SNAPSHOTS_DIR))
)
# Use cache-first snapshot resolution without setting process-wide offline flags;
# the combined notebook also supports FLUX with independent admission checks.

import sys
from pathlib import Path
_TRELLIS_HELPER_ROOT = next(
    (
        root for root in (
            Path.cwd().resolve(),
            *Path.cwd().resolve().parents,
            Path.home() / "OEC",
        )
        if (root / "tools" / "trellis_offline.py").is_file()
    ),
    None,
)
if _TRELLIS_HELPER_ROOT is None:
    raise RuntimeError("TRELLIS offline helper is not available in this OEC checkout.")
if str(_TRELLIS_HELPER_ROOT) not in sys.path:
    sys.path.insert(0, str(_TRELLIS_HELPER_ROOT))
from tools.trellis_offline import offline_hub_mode, prepare_local_trellis_snapshot
TRELLIS_SNAPSHOT_OVERLAY_TEMP = None
import torch
import random
import gc
from PIL import Image
import gradio as gr

from huggingface_hub import snapshot_download
try:
    from huggingface_hub.errors import LocalEntryNotFoundError
except ImportError:
    from huggingface_hub.utils import LocalEntryNotFoundError

from purge_manager import DiskPurgeManager
from storage_manager import managed_storage_operation, storage_preflight
from storage_policy import BYTES_PER_GB, estimate_hf_repo_bytes

ATTN_BACKEND_REQUESTED = os.environ.get("AIEROS_TRELLIS_ATTN_BACKEND", "xformers")
if ATTN_BACKEND_REQUESTED not in ("xformers", "flash_attn"):
    raise ValueError(
        f"Unsupported TRELLIS sparse-attention backend {ATTN_BACKEND_REQUESTED!r}; "
        "use xformers or flash_attn (SDPA is not supported by TRELLIS sparse attention)."
    )
os.environ["ATTN_BACKEND"] = ATTN_BACKEND_REQUESTED
os.environ["SPARSE_ATTN_BACKEND"] = ATTN_BACKEND_REQUESTED
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ["TRELLIS_CACHE_DIR"] = os.path.join(SYSTEM_DIRS['models'], "trellis_weights")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Probe the actual CUDA implementation before importing TRELLIS, so a mismatched
# wheel fails here with an actionable message rather than halfway through generation.
if ATTN_BACKEND_REQUESTED == "xformers":
    try:
        import xformers.ops as xops
        probe = torch.zeros((1, 2, 1, 64), device="cuda", dtype=torch.float16)
        xops.memory_efficient_attention(probe, probe, probe)
        del probe
        print("[ATTENTION] xFormers CUDA operator probe passed.")
    except Exception as exc:
        raise RuntimeError(
            "The installed xFormers CUDA operator is incompatible with this PyTorch/CUDA build. "
            "Run the TRELLIS setup cell in phase2_3d to install the pinned compatible wheel, "
            "or set AIEROS_TRELLIS_ATTN_BACKEND=flash_attn if flash_attn is installed."
        ) from exc
else:
    import flash_attn
    print("[ATTENTION] Using flash_attn for dense and sparse attention.")

from trellis2.pipelines import Trellis2ImageTo3DPipeline
import o_voxel

OUTPUT_DIR = os.path.join(SYSTEM_DIRS['outputs'], "3d_models")

def _find_cached_trellis_weight_snapshot(hub_cache_dir):
    # Cached TRELLIS snapshots may have weight shards but lack root marker files.
    snapshot_root = os.path.join(hub_cache_dir, "models--microsoft--TRELLIS.2-4B", "snapshots")
    candidates = []
    if os.path.isdir(snapshot_root):
        for name in os.listdir(snapshot_root):
            candidate = os.path.join(snapshot_root, name)
            if not os.path.isdir(candidate):
                continue
            weight_files = []
            for root, _, names in os.walk(candidate):
                weight_files.extend(os.path.join(root, n) for n in names if n.endswith(".safetensors"))
            if weight_files:
                candidates.append((sum(os.path.getsize(p) for p in weight_files), candidate))
    return max(candidates, key=lambda item: item[0])[1] if candidates else None


def load_trellis_pipeline():
    repo_id = TRELLIS_REPO_ID
    hub_cache_dir = os.path.join(SYSTEM_DIRS["models"], "hub")
    try:
        local_model_dir = snapshot_download(
            repo_id=repo_id, cache_dir=hub_cache_dir, local_files_only=True,
        )
        print(f"TRELLIS cache hit; resolved local snapshot: {local_model_dir}")
    except LocalEntryNotFoundError:
        local_model_dir = _find_cached_trellis_weight_snapshot(hub_cache_dir)
        if local_model_dir:
            print(f"TRELLIS_WEIGHT_SNAPSHOT_CACHE_HIT snapshot={local_model_dir}")
        else:
            required_gb = (
                estimate_hf_repo_bytes(repo_id) / BYTES_PER_GB
                + max(0.0, float(os.environ.get("AIEROS_MODEL_DOWNLOAD_SAFETY_GB", "2.0")))
            )
            min_free_gb = float(os.environ.get("AIEROS_MODEL_MIN_FREE_GB", "2.0"))
            ok, reason = storage_preflight(
                "models", required_gb=required_gb, require_fresh_quota=True,
                minimum_free_gb=min_free_gb,
            )
            if not ok:
                raise RuntimeError(f"TRELLIS download denied by storage policy: {reason}")
            with managed_storage_operation(
                "models", required_gb=required_gb, require_fresh_quota=True,
                minimum_free_gb=min_free_gb,
            ):
                ok, reason = storage_preflight(
                    "models", required_gb=required_gb, require_fresh_quota=True,
                    minimum_free_gb=min_free_gb,
                )
                if not ok:
                    raise RuntimeError(f"TRELLIS download denied inside managed operation: {reason}")
                local_model_dir = snapshot_download(repo_id=repo_id, cache_dir=hub_cache_dir)
    global TRELLIS_SNAPSHOT_OVERLAY_TEMP
    _trellis_cache_roots = [
        hub_cache_dir,
        SYSTEM_DIRS["models"],
        Path.home() / ".cache" / "huggingface" / "hub",
    ]
    for _trellis_cache_env in ("HF_HUB_CACHE", "HUGGINGFACE_HUB_CACHE", "TRANSFORMERS_CACHE"):
        _trellis_cache_path = os.environ.get(_trellis_cache_env)
        if _trellis_cache_path:
            _trellis_cache_roots.append(Path(_trellis_cache_path))
    local_model_dir, TRELLIS_SNAPSHOT_OVERLAY_TEMP = prepare_local_trellis_snapshot(
        local_model_dir, repo_id=repo_id, cache_roots=tuple(_trellis_cache_roots)
    )
    with offline_hub_mode(TRELLIS_SNAPSHOT_OVERLAY_TEMP):
        return Trellis2ImageTo3DPipeline.from_pretrained(local_model_dir)


trellis_pipe = load_trellis_pipeline()
trellis_pipe.cuda()

if ENABLE_DISK_PURGE and os.environ.get("AIEROS_ALLOW_SHARED_MODEL_PURGE") == "1":
    print("\n[WARNING] Explicit shared-cache purge requested. This is unsafe if any other process may still be loading the same model.")
    DiskPurgeManager.purge_hf_model("microsoft/TRELLIS.2-4B")
else:
    print("[STORAGE] Shared model/cache files retained. Set ENABLE_DISK_PURGE=True and AIEROS_ALLOW_SHARED_MODEL_PURGE=1 only after confirming no concurrent loaders.")
# TRELLIS_CACHE_DIR is deliberately retained because pipeline components may use it lazily.

@torch.inference_mode()
def extrude_3d_asset(input_image, decimation_target: int):
    if input_image is None:
        raise gr.Error("Please upload an image first!")
        
    seed = random.randint(0, 999999)
    print(f"\n[EXTRUDING] Processing uploaded image | Seed: {seed}")
    
    try:
        input_image = input_image.convert("RGB")
        width, height = input_image.size
        max_size = 1024
        if max(width, height) > max_size:
            scale = max_size / float(max(width, height))
            width = max(1, int(width * scale))
            height = max(1, int(height * scale))
            input_image = input_image.resize((width, height), Image.Resampling.LANCZOS)

        # TRELLIS.2 works more predictably with a square image aligned to 64px.
        square_size = max(64, ((max(width, height) + 63) // 64) * 64)
        square_image = Image.new("RGB", (square_size, square_size), (255, 255, 255))
        square_image.paste(input_image, ((square_size - width) // 2, (square_size - height) // 2))
        input_image = square_image
        ok, reason = storage_preflight(
            "outputs", required_gb=0.20, require_fresh_quota=True, minimum_free_gb=0.05,
        )
        if not ok:
            raise RuntimeError(f"TRELLIS generation refused before inference by storage policy: {reason}")
        torch.cuda.empty_cache()
        mesh = trellis_pipe.run(input_image)[0]
        mesh.simplify(16777216)

        glb_path = os.path.join(OUTPUT_DIR, f"standalone_t2_{seed}.glb")
        glb = o_voxel.postprocess.to_glb(
            vertices            =   mesh.vertices,
            faces               =   mesh.faces,
            attr_volume         =   mesh.attrs,
            coords              =   mesh.coords,
            attr_layout         =   mesh.layout,
            voxel_size          =   mesh.voxel_size,
            aabb                =   [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
            decimation_target   =   int(decimation_target),
            texture_size        =   4096,
            remesh              =   True,
            remesh_band         =   1,
            remesh_project      =   0,
            verbose             =   False
        )
        ok, reason = storage_preflight(
            "outputs",
            required_gb=0.20,
            require_fresh_quota=True,
            minimum_free_gb=0.05,
        )
        if not ok:
            raise RuntimeError(f"Storage admission denied before managed write: {reason}")
        with managed_storage_operation(
            "outputs",
            required_gb=0.20,
            require_fresh_quota=True,
            minimum_free_gb=0.05,
        ):
            ok, reason = storage_preflight(
                "outputs",
                required_gb=0.20,
                require_fresh_quota=True,
                minimum_free_gb=0.05,
            )
            if not ok:
                raise RuntimeError(f"Storage admission denied inside managed write: {reason}")
            os.makedirs(OUTPUT_DIR, exist_ok=True)
            glb.export(glb_path, extension_webp=True)
        if not os.path.isfile(glb_path) or os.path.getsize(glb_path) <= 20:
            raise RuntimeError(f"TRELLIS export failed to create a non-empty GLB: {glb_path}")
        with open(glb_path, "rb") as glb_file:
            if glb_file.read(4) != b"glTF":
                raise RuntimeError(f"TRELLIS export has an invalid GLB header: {glb_path}")

        del mesh, glb
        gc.collect()
        torch.cuda.empty_cache()
        return glb_path
    except Exception as exc:
        error_trace = traceback.format_exc()
        print(f"\\n[TRELLIS GENERATION ERROR]\\n{error_trace}")
        gc.collect()
        torch.cuda.empty_cache()
        raise gr.Error(f"TRELLIS generation failed: {type(exc).__name__}: {exc}") from exc

with gr.Blocks(title="TRELLIS.2 Standalone", theme=gr.themes.Monochrome()) as demo_trellis:
    gr.Markdown("# 🕹️ TRELLIS.2-4B (Isolated Testing)")
    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(label="Upload 2D Concept", type="pil")
            decimation = gr.Slider(label="Target Polygons", minimum=10000, maximum=500000, value=100000, step=10000)
            generate_btn = gr.Button("Extrude to 3D", variant="primary", size="lg")
        with gr.Column(scale=2):
            output_3d = gr.Model3D(label="3D Interactive Viewer")

    generate_btn.click(fn=extrude_3d_asset, inputs=[input_image, decimation], outputs=[output_3d])

print("Starting TRELLIS Server on port 7862...")
demo_trellis.launch(server_name="0.0.0.0", server_port=7862, inline=False, share=False, show_error=True)

# Step 4: FLUX.2 (Interactive Studio - Text to Image)

In [ ]:
import core_config
from core_config import SYSTEM_DIRS

import os
import torch
import random
import gc
import gradio as gr
from diffusers import DiffusionPipeline
# Make this studio cell self-contained when re-running it after the setup cells.
from huggingface_hub import snapshot_download
try:
    from huggingface_hub.errors import LocalEntryNotFoundError
except ImportError:
    from huggingface_hub.utils import LocalEntryNotFoundError

from storage_manager import managed_storage_operation, storage_preflight
from storage_policy import BYTES_PER_GB, estimate_hf_repo_bytes

OUTPUT_DIR = os.path.join(SYSTEM_DIRS['outputs'], "images")

def load_flux_pipeline():
    model_id = "black-forest-labs/FLUX.2-klein-4B"
    primary_cache_dir = os.path.join(SYSTEM_DIRS["models"], "hub")
    # Existing FLUX weights may be cached directly under aireos_models rather than hub/.
    local_model_dir = None
    for cache_root in (SYSTEM_DIRS["models"], primary_cache_dir):
        try:
            local_model_dir = snapshot_download(
                repo_id=model_id, cache_dir=cache_root, local_files_only=True,
            )
            print(f"FLUX_CACHE_HIT cache_root={cache_root} snapshot={local_model_dir}")
            break
        except LocalEntryNotFoundError:
            continue
    if local_model_dir is None:
        required_gb = (
            estimate_hf_repo_bytes(model_id) / BYTES_PER_GB
            + max(0.0, float(os.environ.get("AIEROS_MODEL_DOWNLOAD_SAFETY_GB", "2.0")))
        )
        min_free_gb = float(os.environ.get("AIEROS_MODEL_MIN_FREE_GB", "2.0"))
        ok, reason = storage_preflight(
            "models", required_gb=required_gb, require_fresh_quota=True, minimum_free_gb=min_free_gb,
        )
        if not ok:
            raise RuntimeError(f"FLUX download denied by storage policy: {reason}")
        with managed_storage_operation(
            "models", required_gb=required_gb, require_fresh_quota=True, minimum_free_gb=min_free_gb,
        ):
            ok, reason = storage_preflight(
                "models", required_gb=required_gb, require_fresh_quota=True, minimum_free_gb=min_free_gb,
            )
            if not ok:
                raise RuntimeError(f"FLUX download denied inside managed operation: {reason}")
            local_model_dir = snapshot_download(repo_id=model_id, cache_dir=primary_cache_dir)
    return DiffusionPipeline.from_pretrained(local_model_dir, **kwargs)


flux_pipe = load_flux_pipeline()

print("\n[STORAGE] Shared FLUX model/cache purge disabled; cached weights are retained.")

@torch.inference_mode()
def generate_concept(prompt: str, seed: int, randomize: bool):
    if randomize: seed = random.randint(0, 999999)
    print(f"\n[GENERATING] Prompt: '{prompt}' | Seed: {seed}")
    
    ok, reason = storage_preflight(
        "outputs", required_gb=0.05, require_fresh_quota=True, minimum_free_gb=0.02,
    )
    if not ok:
        raise RuntimeError(f"FLUX generation refused before inference by storage policy: {reason}")
    generator = torch.Generator(device="cpu").manual_seed(seed)
    image = flux_pipe(
        prompt,
        guidance_scale=3.5, 
        num_inference_steps=28, 
        max_sequence_length=512,
        generator=generator
    ).images[0].convert("RGB")
    
    out_path = os.path.join(OUTPUT_DIR, f"flux2_concept_{seed}.png")
    ok, reason = storage_preflight(
        "outputs",
        required_gb=0.05,
        require_fresh_quota=True,
        minimum_free_gb=0.02,
    )
    if not ok:
        raise RuntimeError(f"Storage admission denied before managed write: {reason}")
    with managed_storage_operation(
        "outputs",
        required_gb=0.05,
        require_fresh_quota=True,
        minimum_free_gb=0.02,
    ):
        ok, reason = storage_preflight(
            "outputs",
            required_gb=0.05,
            require_fresh_quota=True,
            minimum_free_gb=0.02,
        )
        if not ok:
            raise RuntimeError(f"Storage admission denied inside managed write: {reason}")
        image.save(out_path)
    if not os.path.isfile(out_path) or os.path.getsize(out_path) <= 8:
        raise RuntimeError(f"FLUX export failed to create a non-empty PNG: {out_path}")
    with open(out_path, "rb") as image_file:
        if image_file.read(8) != b"\x89PNG\r\n\x1a\n":
            raise RuntimeError(f"FLUX export has an invalid PNG header: {out_path}")
    gc.collect()
    torch.cuda.empty_cache()
    return image, seed

with gr.Blocks(title="FLUX.2 Standalone", theme=gr.themes.Monochrome()) as demo_flux:
    gr.Markdown("# 🎨 FLUX.2-4B (Isolated Testing)")
    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(label="Prompt", value="A sci-fi crate", lines=4)
            seed = gr.Number(label="Seed", value=42, precision=0)
            randomize = gr.Checkbox(label="Randomize", value=True)
            generate_btn = gr.Button("Generate", variant="primary")
        with gr.Column(scale=2):
            output_image = gr.Image(label="FLUX.2 Output")

    generate_btn.click(fn=generate_concept, inputs=[prompt, seed, randomize], outputs=[output_image, seed])

print("Starting FLUX Server on port 7863...")
demo_flux.launch(server_name="0.0.0.0", server_port=7863, inline=False, share=False)

In [ ]:
# demo.close()

In [ ]:
!nvidia-smi